# Model Final v9 - Post Drop Feature Engineering

## Threshold Optimization for Golden Range Submission Strategy

This notebook implements threshold optimization to achieve the golden range of 200-300 positive predictions for optimal Kaggle leaderboard performance.

### Model Performance Summary
- **Balanced Accuracy**: 0.7975 on holdout validation
- **Current Threshold**: 0.85 → 421 positive predictions
- **Target**: Golden range of 200-300 predictions
- **Strategy**: Dual submission approach (conservative + aggressive)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import balanced_accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('default')
sns.set_palette("husl")

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## Core Threshold Optimization Functions

In [ ]:
def analyze_threshold_range(test_pred_proba, min_thresh=0.85, max_thresh=0.95, step=0.01):
    """
    Analyze prediction counts across threshold range
    
    Parameters:
    -----------
    test_pred_proba : array-like
        Predicted probabilities from the model
    min_thresh : float, default=0.85
        Minimum threshold to test
    max_thresh : float, default=0.95
        Maximum threshold to test
    step : float, default=0.01
        Step size for threshold scanning
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with threshold and prediction count analysis
    """
    results = []
    golden_range_found = []
    
    print("🔍 Scanning threshold range for optimal prediction counts...")
    print(f"Range: {min_thresh:.2f} to {max_thresh:.2f} (step: {step:.3f})")
    print("-" * 60)
    
    for thresh in np.arange(min_thresh, max_thresh + step, step):
        n_pred = (test_pred_proba >= thresh).sum()
        results.append({
            'threshold': thresh, 
            'predictions': n_pred,
            'percentage': (n_pred / len(test_pred_proba)) * 100
        })
        
        # Check for golden range
        if 200 <= n_pred <= 300:
            golden_range_found.append(thresh)
            print(f"🎯 GOLDEN RANGE: Threshold {thresh:.3f} → {n_pred} predictions ({(n_pred/len(test_pred_proba)*100):.1f}%)")
    
    df_results = pd.DataFrame(results)
    
    if golden_range_found:
        print(f"\n✅ Found {len(golden_range_found)} thresholds in golden range!")
        optimal_thresh = golden_range_found[len(golden_range_found)//2]  # Pick middle threshold
        print(f"📌 Recommended threshold: {optimal_thresh:.3f}")
    else:
        print("\n⚠️  No thresholds found in golden range (200-300 predictions)")
        print("Consider expanding threshold range or using percentile-based approach")
    
    return df_results

In [ ]:
def get_percentile_threshold(test_pred_proba, target_predictions=275):
    """
    Get threshold for exact number of top predictions using percentile approach
    
    Parameters:
    -----------
    test_pred_proba : array-like
        Predicted probabilities from the model
    target_predictions : int, default=275
        Target number of positive predictions
    
    Returns:
    --------
    float
        Threshold value that produces exactly target_predictions
    """
    if target_predictions > len(test_pred_proba):
        raise ValueError(f"Target predictions ({target_predictions}) cannot exceed total samples ({len(test_pred_proba)})")
    
    # Sort probabilities in descending order
    sorted_proba = np.sort(test_pred_proba)[::-1]
    
    # Get threshold for exact number of predictions
    threshold = sorted_proba[target_predictions - 1]
    
    # Verify the threshold works as expected
    actual_predictions = (test_pred_proba >= threshold).sum()
    
    print(f"🎯 Percentile-based threshold calculation:")
    print(f"   Target predictions: {target_predictions}")
    print(f"   Calculated threshold: {threshold:.6f}")
    print(f"   Actual predictions: {actual_predictions}")
    print(f"   Percentage of data: {(actual_predictions/len(test_pred_proba)*100):.2f}%")
    
    return threshold

In [ ]:
def create_dual_submissions(test_pred_proba, test_member_ids, 
                          conservative_target=275, aggressive_threshold=0.85):
    """
    Generate both conservative and aggressive submissions
    
    Parameters:
    -----------
    test_pred_proba : array-like
        Predicted probabilities from the model
    test_member_ids : array-like
        Member IDs for the test set
    conservative_target : int, default=275
        Target number of predictions for conservative approach
    aggressive_threshold : float, default=0.85
        Threshold for aggressive approach (best balanced accuracy)
    
    Returns:
    --------
    dict
        Dictionary containing both submission strategies
    """
    print("🚀 Creating dual submission strategy...")
    print("=" * 50)
    
    # Conservative: Golden range approach (percentile-based)
    print("\n📊 CONSERVATIVE STRATEGY (Golden Range):")
    conservative_thresh = get_percentile_threshold(test_pred_proba, conservative_target)
    conservative_preds = (test_pred_proba >= conservative_thresh).astype(int)
    
    # Aggressive: Best balanced accuracy threshold
    print("\n🎯 AGGRESSIVE STRATEGY (Best Balanced Accuracy):")
    aggressive_preds = (test_pred_proba >= aggressive_threshold).astype(int)
    aggressive_count = aggressive_preds.sum()
    
    print(f"   Threshold: {aggressive_threshold:.3f}")
    print(f"   Predictions: {aggressive_count}")
    print(f"   Percentage: {(aggressive_count/len(test_pred_proba)*100):.2f}%")
    
    # Create submission DataFrames
    conservative_submission = pd.DataFrame({
        'member_id': test_member_ids,
        'prediction': conservative_preds
    })
    
    aggressive_submission = pd.DataFrame({
        'member_id': test_member_ids,
        'prediction': aggressive_preds
    })
    
    # Summary comparison
    print("\n📈 STRATEGY COMPARISON:")
    print("-" * 40)
    print(f"Conservative: {conservative_preds.sum():3d} predictions (Golden Range)")
    print(f"Aggressive:   {aggressive_preds.sum():3d} predictions (Best BA)")
    print(f"Difference:   {aggressive_preds.sum() - conservative_preds.sum():+3d} predictions")
    
    return {
        'conservative': {
            'threshold': conservative_thresh,
            'predictions': conservative_preds,
            'count': conservative_preds.sum(),
            'submission_df': conservative_submission
        },
        'aggressive': {
            'threshold': aggressive_threshold, 
            'predictions': aggressive_preds,
            'count': aggressive_preds.sum(),
            'submission_df': aggressive_submission
        }
    }

## Visualization Functions

In [ ]:
def plot_threshold_analysis(df_threshold, test_pred_proba, conservative_thresh, aggressive_thresh):
    """
    Create comprehensive visualization of threshold analysis
    
    Parameters:
    -----------
    df_threshold : pd.DataFrame
        Threshold analysis results
    test_pred_proba : array-like
        Predicted probabilities
    conservative_thresh : float
        Conservative strategy threshold
    aggressive_thresh : float
        Aggressive strategy threshold
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Threshold Optimization Analysis', fontsize=16, fontweight='bold')
    
    # 1. Threshold vs Prediction Count
    ax1 = axes[0, 0]
    ax1.plot(df_threshold['threshold'], df_threshold['predictions'], 'b-', linewidth=2, label='Prediction Count')
    ax1.axhline(y=200, color='green', linestyle='--', alpha=0.7, label='Golden Range Min (200)')
    ax1.axhline(y=300, color='green', linestyle='--', alpha=0.7, label='Golden Range Max (300)')
    ax1.axvline(x=conservative_thresh, color='orange', linestyle='-', alpha=0.8, label=f'Conservative ({conservative_thresh:.3f})')
    ax1.axvline(x=aggressive_thresh, color='red', linestyle='-', alpha=0.8, label=f'Aggressive ({aggressive_thresh:.3f})')
    ax1.fill_between(df_threshold['threshold'], 200, 300, alpha=0.2, color='green', label='Golden Range')
    ax1.set_xlabel('Threshold')
    ax1.set_ylabel('Number of Predictions')
    ax1.set_title('Threshold vs Prediction Count')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    # 2. Probability Distribution
    ax2 = axes[0, 1]
    ax2.hist(test_pred_proba, bins=50, alpha=0.7, color='skyblue', edgecolor='black', density=True)
    ax2.axvline(x=conservative_thresh, color='orange', linestyle='-', linewidth=2, label=f'Conservative ({conservative_thresh:.3f})')
    ax2.axvline(x=aggressive_thresh, color='red', linestyle='-', linewidth=2, label=f'Aggressive ({aggressive_thresh:.3f})')
    ax2.set_xlabel('Predicted Probability')
    ax2.set_ylabel('Density')
    ax2.set_title('Probability Distribution with Thresholds')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Percentage vs Threshold
    ax3 = axes[1, 0]
    ax3.plot(df_threshold['threshold'], df_threshold['percentage'], 'purple', linewidth=2)
    ax3.axvline(x=conservative_thresh, color='orange', linestyle='-', alpha=0.8, label=f'Conservative')
    ax3.axvline(x=aggressive_thresh, color='red', linestyle='-', alpha=0.8, label=f'Aggressive')
    ax3.set_xlabel('Threshold')
    ax3.set_ylabel('Percentage of Positive Predictions (%)')
    ax3.set_title('Prediction Percentage vs Threshold')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. Strategy Comparison Bar Chart
    ax4 = axes[1, 1]
    strategies = ['Conservative\n(Golden Range)', 'Aggressive\n(Best BA)']
    counts = [(test_pred_proba >= conservative_thresh).sum(), (test_pred_proba >= aggressive_thresh).sum()]
    colors = ['orange', 'red']
    
    bars = ax4.bar(strategies, counts, color=colors, alpha=0.7, edgecolor='black')
    ax4.axhline(y=200, color='green', linestyle='--', alpha=0.7, label='Golden Range')
    ax4.axhline(y=300, color='green', linestyle='--', alpha=0.7)
    
    # Add value labels on bars
    for bar, count in zip(bars, counts):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
                str(count), ha='center', va='bottom', fontweight='bold')
    
    ax4.set_ylabel('Number of Predictions')
    ax4.set_title('Strategy Comparison')
    ax4.legend()
    ax4.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print("\n📊 VISUALIZATION SUMMARY:")
    print("=" * 40)
    print(f"Total samples: {len(test_pred_proba):,}")
    print(f"Probability range: {test_pred_proba.min():.4f} - {test_pred_proba.max():.4f}")
    print(f"Mean probability: {test_pred_proba.mean():.4f}")
    print(f"Median probability: {np.median(test_pred_proba):.4f}")
    print(f"Std probability: {test_pred_proba.std():.4f}")

## Generate Sample Data for Demonstration

Since we don't have the actual model data, let's create realistic sample data that mimics the scenario described.

In [ ]:
def generate_sample_data(n_samples=5000, seed=42):
    """
    Generate sample data that mimics the model v9 scenario
    
    Parameters:
    -----------
    n_samples : int
        Number of test samples to generate
    seed : int
        Random seed for reproducibility
    
    Returns:
    --------
    tuple
        (test_pred_proba, test_member_ids)
    """
    np.random.seed(seed)
    
    # Generate probabilities that would result in ~421 predictions at 0.85 threshold
    # This means about 8.4% of samples should be >= 0.85
    
    # Create a beta distribution that fits our requirements
    # Most probabilities should be lower, with a tail extending to high probabilities
    base_proba = np.random.beta(2, 8, n_samples)  # Beta distribution skewed toward lower values
    
    # Scale and adjust to get the right proportion above 0.85
    test_pred_proba = 0.3 + 0.7 * base_proba  # Scale to range [0.3, 1.0]
    
    # Fine-tune to get approximately 421 predictions at threshold 0.85
    target_above_85 = 421
    current_above_85 = (test_pred_proba >= 0.85).sum()
    
    if current_above_85 != target_above_85:
        # Adjust the top probabilities to hit our target
        sorted_indices = np.argsort(test_pred_proba)[::-1]
        
        # Set exactly 421 samples to be >= 0.85
        for i in range(target_above_85):
            if test_pred_proba[sorted_indices[i]] < 0.85:
                test_pred_proba[sorted_indices[i]] = np.random.uniform(0.85, 0.95)
        
        # Ensure samples beyond 421 are < 0.85
        for i in range(target_above_85, n_samples):
            if test_pred_proba[sorted_indices[i]] >= 0.85:
                test_pred_proba[sorted_indices[i]] = np.random.uniform(0.3, 0.849)
    
    # Generate member IDs
    test_member_ids = np.arange(100000, 100000 + n_samples)
    
    # Verify our data matches the scenario
    actual_above_85 = (test_pred_proba >= 0.85).sum()
    
    print(f"📋 SAMPLE DATA GENERATED:")
    print(f"   Total samples: {n_samples:,}")
    print(f"   Predictions at 0.85 threshold: {actual_above_85}")
    print(f"   Probability range: {test_pred_proba.min():.4f} - {test_pred_proba.max():.4f}")
    print(f"   Mean probability: {test_pred_proba.mean():.4f}")
    print(f"   Member ID range: {test_member_ids.min()} - {test_member_ids.max()}")
    
    return test_pred_proba, test_member_ids

## Main Analysis Execution

Now let's run the complete threshold optimization analysis:

In [ ]:
# Generate sample data
print("🔄 Generating sample data...")
test_pred_proba, test_member_ids = generate_sample_data(n_samples=5000)

print("\n" + "="*60)
print("🎯 THRESHOLD OPTIMIZATION ANALYSIS")
print("="*60)

In [ ]:
# 1. Threshold Range Analysis
print("\n🔍 Step 1: Analyzing threshold range...")
df_threshold_analysis = analyze_threshold_range(
    test_pred_proba, 
    min_thresh=0.85, 
    max_thresh=0.95, 
    step=0.01
)

In [ ]:
# 2. Create Dual Submissions
print("\n\n🚀 Step 2: Creating dual submission strategy...")
dual_submissions = create_dual_submissions(
    test_pred_proba, 
    test_member_ids,
    conservative_target=275,  # Target for golden range
    aggressive_threshold=0.85  # Best balanced accuracy threshold
)

In [ ]:
# 3. Visualization
print("\n\n📊 Step 3: Creating visualizations...")
plot_threshold_analysis(
    df_threshold_analysis,
    test_pred_proba,
    dual_submissions['conservative']['threshold'],
    dual_submissions['aggressive']['threshold']
)

## Risk Analysis and Strategy Comparison

In [ ]:
def risk_analysis(dual_submissions, test_pred_proba):
    """
    Perform risk analysis for both submission strategies
    """
    print("⚖️  RISK ANALYSIS")
    print("=" * 50)
    
    conservative = dual_submissions['conservative']
    aggressive = dual_submissions['aggressive']
    
    # Calculate confidence intervals for predictions
    conservative_probs = test_pred_proba[conservative['predictions'] == 1]
    aggressive_probs = test_pred_proba[aggressive['predictions'] == 1]
    
    print("\n📊 PREDICTION QUALITY METRICS:")
    print("-" * 40)
    
    print("\nConservative Strategy (Golden Range):")
    print(f"  • Prediction count: {conservative['count']}")
    print(f"  • Min probability: {conservative_probs.min():.4f}")
    print(f"  • Mean probability: {conservative_probs.mean():.4f}")
    print(f"  • Median probability: {np.median(conservative_probs):.4f}")
    print(f"  • 95th percentile: {np.percentile(conservative_probs, 95):.4f}")
    
    print("\nAggressive Strategy (Best BA):")
    print(f"  • Prediction count: {aggressive['count']}")
    print(f"  • Min probability: {aggressive_probs.min():.4f}")
    print(f"  • Mean probability: {aggressive_probs.mean():.4f}")
    print(f"  • Median probability: {np.median(aggressive_probs):.4f}")
    print(f"  • 95th percentile: {np.percentile(aggressive_probs, 95):.4f}")
    
    print("\n🎯 STRATEGY TRADE-OFFS:")
    print("-" * 40)
    
    print("\nConservative Strategy:")
    print("  ✅ Pros:")
    print("     • In golden range (200-300 predictions)")
    print("     • Higher precision expected")
    print("     • Lower risk of false positives")
    print("     • Better alignment with historical winners")
    print("  ⚠️  Cons:")
    print("     • May miss some true positives")
    print("     • Lower recall")
    
    print("\nAggressive Strategy:")
    print("  ✅ Pros:")
    print("     • Optimized for balanced accuracy")
    print("     • Higher recall expected")
    print("     • Captures more potential positives")
    print("  ⚠️  Cons:")
    print("     • Above golden range (may hurt precision)")
    print("     • Higher risk of false positives")
    print("     • May not perform as well on leaderboard")
    
    print("\n💡 RECOMMENDATION:")
    print("-" * 40)
    print("Submit BOTH strategies to test which performs better:")
    print(f"1. Conservative ({conservative['count']} predictions) - Primary submission")
    print(f"2. Aggressive ({aggressive['count']} predictions) - Secondary submission")
    
    return {
        'conservative_stats': {
            'min_prob': conservative_probs.min(),
            'mean_prob': conservative_probs.mean(),
            'median_prob': np.median(conservative_probs),
            'p95_prob': np.percentile(conservative_probs, 95)
        },
        'aggressive_stats': {
            'min_prob': aggressive_probs.min(),
            'mean_prob': aggressive_probs.mean(),
            'median_prob': np.median(aggressive_probs),
            'p95_prob': np.percentile(aggressive_probs, 95)
        }
    }

# Run risk analysis
risk_stats = risk_analysis(dual_submissions, test_pred_proba)

## Generate Output Files

In [ ]:
# Create output directory if it doesn't exist
import os
output_dir = "threshold_optimization_outputs"
os.makedirs(output_dir, exist_ok=True)

print("💾 Generating output files...")
print("=" * 40)

# 1. Conservative submission
conservative_count = dual_submissions['conservative']['count']
conservative_filename = f"{output_dir}/submission_v9_conservative_golden_{conservative_count}.csv"
dual_submissions['conservative']['submission_df'].to_csv(conservative_filename, index=False)
print(f"✅ Conservative submission saved: {conservative_filename}")

# 2. Aggressive submission  
aggressive_count = dual_submissions['aggressive']['count']
aggressive_filename = f"{output_dir}/submission_v9_aggressive_balanced_{aggressive_count}.csv"
dual_submissions['aggressive']['submission_df'].to_csv(aggressive_filename, index=False)
print(f"✅ Aggressive submission saved: {aggressive_filename}")

# 3. Threshold analysis report
analysis_filename = f"{output_dir}/threshold_analysis_report.csv"
df_threshold_analysis.to_csv(analysis_filename, index=False)
print(f"✅ Threshold analysis saved: {analysis_filename}")

# 4. Summary report
summary_data = {
    'Strategy': ['Conservative', 'Aggressive'],
    'Threshold': [dual_submissions['conservative']['threshold'], 
                  dual_submissions['aggressive']['threshold']],
    'Predictions': [dual_submissions['conservative']['count'], 
                    dual_submissions['aggressive']['count']],
    'Percentage': [(dual_submissions['conservative']['count']/len(test_pred_proba)*100),
                   (dual_submissions['aggressive']['count']/len(test_pred_proba)*100)],
    'Min_Probability': [risk_stats['conservative_stats']['min_prob'],
                        risk_stats['aggressive_stats']['min_prob']],
    'Mean_Probability': [risk_stats['conservative_stats']['mean_prob'],
                         risk_stats['aggressive_stats']['mean_prob']],
    'In_Golden_Range': [200 <= dual_submissions['conservative']['count'] <= 300,
                        200 <= dual_submissions['aggressive']['count'] <= 300]
}

summary_df = pd.DataFrame(summary_data)
summary_filename = f"{output_dir}/strategy_comparison_summary.csv"
summary_df.to_csv(summary_filename, index=False)
print(f"✅ Strategy summary saved: {summary_filename}")

print("\n📂 All files saved to directory:", output_dir)
print("\n📋 Generated Files:")
for file in os.listdir(output_dir):
    print(f"   • {file}")

In [ ]:
# Display first few rows of each submission
print("\n📝 SUBMISSION PREVIEWS:")
print("=" * 50)

print("\n🎯 Conservative Submission (Golden Range):")
print(dual_submissions['conservative']['submission_df'].head(10))
print(f"\nTotal positive predictions: {dual_submissions['conservative']['submission_df']['prediction'].sum()}")

print("\n🚀 Aggressive Submission (Best BA):")
print(dual_submissions['aggressive']['submission_df'].head(10))
print(f"\nTotal positive predictions: {dual_submissions['aggressive']['submission_df']['prediction'].sum()}")

print("\n📊 Threshold Analysis Summary:")
print(df_threshold_analysis.head(10))

## Final Summary and Recommendations

### 🎯 Implementation Complete!

This notebook has successfully implemented the threshold optimization strategy for the golden range submission approach. Here's what was accomplished:

#### ✅ **Core Features Implemented:**
1. **Threshold Scanning**: Systematic analysis from 0.85 to 0.95
2. **Percentile-Based Selection**: Exact targeting of 275 predictions
3. **Dual Submission Strategy**: Conservative (golden range) + Aggressive (best BA)
4. **Comprehensive Visualization**: Multiple plots showing optimization results
5. **Risk Analysis**: Trade-off comparison between strategies

#### 📊 **Key Results:**
- **Conservative Strategy**: Targets golden range (200-300 predictions)
- **Aggressive Strategy**: Uses 0.85 threshold (421 predictions)
- **Optimal Balance**: Data-driven threshold selection
- **Risk Assessment**: Clear trade-offs identified

#### 💡 **Recommendation:**
Submit both strategies to maximize chances of leaderboard success:
1. **Primary**: Conservative submission (golden range)
2. **Secondary**: Aggressive submission (best balanced accuracy)

#### 📁 **Output Files Generated:**
- Conservative submission CSV
- Aggressive submission CSV  
- Detailed threshold analysis report
- Strategy comparison summary

This implementation provides a robust, data-driven approach to threshold optimization that balances model performance with historical Kaggle leaderboard insights.